# diagram overview

## Environment
Install project dependencies once with `python -m pip install -r requirements.txt`.
The source dataset is not checked into this repository.

In [ ]:
from pathlib import Path


In [ ]:
data_path = Path("data/restaurants.csv")
if not data_path.is_file():
    raise FileNotFoundError("Place the source dataset at data/restaurants.csv before running this notebook.")
df = pd.read_csv(data_path)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.notnull().sum()


In [ ]:
df.head()

In [ ]:
for column in df.columns:
  unique_values=df[column].unique()
  print(f'uique value in {column}: ',unique_values)

In [ ]:

fig,ax=plt.subplots()
sns.histplot(df['stars_user'],kde=True)
plt.title('stars_user')

In [ ]:

fig,ax=plt.subplots()
sns.histplot(df['date'],kde=True)
plt.title('date')

In [ ]:
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
print(df.info())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# User vs Business ratings
sns.histplot(data=df, x='stars_user', bins=20, ax=axes[0], kde=True)
axes[0].set_title('Distribution of User Given Ratings', fontsize=14, fontweight='bold')

sns.histplot(data=df, x='stars_business', bins=20, ax=axes[1], kde=True)
axes[1].set_title('Distribution of Business Average Ratings', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Rating comparison
fig = plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='stars_business', y='stars_user', alpha=0.6)
plt.title('User Rating vs Business Average Rating', fontsize=14, fontweight='bold')
plt.xlabel('Business Average Stars')
plt.ylabel('User Given Stars')
plt.show()

In [ ]:
# Review reaction analysis - Better layout
reaction_cols = ['useful_review', 'funny_review', 'cool_review']
reaction_data = df[reaction_cols]

# Calculate correlation matrix with the correct column name
rating_cols = ['stars_user']  # Changed from 'stars' to 'stars_user'
corr_cols = rating_cols + reaction_cols
corr_matrix = df[corr_cols].corr()

# Create 2x3 grid for better organization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Pie chart - top left
reaction_totals = reaction_data.sum()
axes[0,0].pie(reaction_totals, labels=reaction_totals.index, autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Distribution of Review Reactions', fontweight='bold')

# Correlation heatmap - span two columns
axes[0,1].axis('off')  # Hide the middle top subplot
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[0,2])
axes[0,2].set_title('Correlation: Ratings vs Reactions', fontweight='bold')

# Individual reaction distributions - bottom row
for i, col in enumerate(reaction_cols):
    sns.histplot(data=df, x=col, ax=axes[1,i], bins=30, kde=True)
    axes[1,i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[1,i].set_xlim(0, df[col].quantile(0.95))

plt.tight_layout()
plt.show()

In [ ]:
# Convert date columns
df['date'] = pd.to_datetime(df['date'])
df['yelping_since'] = pd.to_datetime(df['yelping_since'])

# User tenure analysis
df['user_tenure_days'] = (df['date'] - df['yelping_since']).dt.days

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# User tenure vs rating behavior
sns.scatterplot(data=df, x='user_tenure_days', y='stars_user', alpha=0.5, ax=axes[0,0])
axes[0,0].set_title('User Tenure vs Rating Given', fontweight='bold')

# Review activity over years
df['review_year'] = df['date'].dt.year
yearly_reviews = df['review_year'].value_counts().sort_index()
axes[0,1].plot(yearly_reviews.index, yearly_reviews.values, marker='o', linewidth=2)
axes[0,1].set_title('Review Activity Over Time', fontweight='bold')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Number of Reviews')

# User experience level vs rating
sns.boxplot(data=df, x=pd.cut(df['review_count_user'], bins=5), y='stars_user', ax=axes[1,0])
axes[1,0].set_title('Rating Behavior by User Experience Level', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=45)

# User compliment analysis
compliment_cols = [col for col in df.columns if 'compliment' in col]
compliment_totals = df[compliment_cols].sum().sort_values(ascending=False)
sns.barplot(x=compliment_totals.values, y=compliment_totals.index, ax=axes[1,1])
axes[1,1].set_title('Distribution of User Compliments', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Business rating vs review count
sns.scatterplot(data=df, x='review_count_business', y='stars_business', alpha=0.6, ax=axes[0,0])
axes[0,0].set_title('Business Rating vs Number of Reviews', fontweight='bold')

# Open vs closed businesses rating distribution
sns.boxplot(data=df, x='is_open', y='stars_business', ax=axes[0,1])
axes[0,1].set_title('Rating Distribution: Open vs Closed Businesses', fontweight='bold')
axes[0,1].set_xticklabels(['Closed', 'Open'])

# Check-in and tip analysis
business_metrics = df[['checkin_count', 'tip_count', 'stars_business']].corr()
sns.heatmap(business_metrics, annot=True, cmap='viridis', ax=axes[1,0])
axes[1,0].set_title('Business Metrics Correlation', fontweight='bold')

# Review count distribution for businesses
sns.histplot(data=df, x='review_count_business', bins=50, ax=axes[1,1], kde=True)
axes[1,1].set_title('Distribution of Business Review Counts', fontweight='bold')
axes[1,1].set_xlim(0, df['review_count_business'].quantile(0.95))

plt.tight_layout()
plt.show()

In [ ]:
# Create user segments based on activity and influence
conditions = [
    (df['review_count_user'] <= 5) & (df['fans'] <= 1),
    (df['review_count_user'] <= 20) & (df['fans'] <= 5),
    (df['review_count_user'] > 20) | (df['fans'] > 5),
    (df['review_count_user'] > 50) | (df['fans'] > 20)
]
segments = ['Casual', 'Regular', 'Active', 'Influencer']
df['user_segment'] = np.select(conditions, segments, default='Influencer')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Segment distribution
segment_counts = df['user_segment'].value_counts()
axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%')
axes[0].set_title('User Segments Distribution', fontweight='bold')

# Average rating by segment
sns.boxplot(data=df, x='user_segment', y='stars_user', order=segments, ax=axes[1])
axes[1].set_title('Rating Behavior by User Segment', fontweight='bold')

# Engagement metrics by segment
engagement_by_segment = df.groupby('user_segment')[['useful_user', 'funny_user', 'cool_user']].mean()
engagement_by_segment.plot(kind='bar', ax=axes[2])
axes[2].set_title('Average Engagement by User Segment', fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

# Your existing code
fig = px.scatter(df, x='review_count_user', y='fans', 
                 color='user_segment', size='average_stars',
                 hover_data=['name_user', 'yelping_since'],
                 title='User Experience vs Influence',
                 labels={'review_count_user': 'Review Count', 'fans': 'Number of Fans'})
fig.show()

In [ ]:
# Select numerical columns for correlation analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Comprehensive Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Prepare data for business clustering
business_features = ['stars_business', 'review_count_business', 'checkin_count', 'tip_count']
business_data = df[business_features].dropna()

# Standardize features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(business_data)

# Apply PCA for visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_data)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], 
                     c=df.loc[business_data.index, 'stars_business'], 
                     alpha=0.6, cmap='viridis')
plt.colorbar(scatter, label='Business Stars')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Business Clustering based on Performance Metrics', fontweight='bold')
plt.show()

In [ ]:
from ydata_profiling import ProfileReport

# Generate the profile report with minimal configuration
prof = ProfileReport(df, 
                    title='Restaurant Health Data Analysis',
                    minimal=True,  # Use minimal configuration to reduce complexity
                    plot={'image_format': 'png'},  # Use PNG instead of SVG
                    progress_bar=False)  # Disable progress bar to reduce dependencies
prof.to_file('output.html')

### Profiling dependencies
Install the packages listed in requirements.txt in the active environment before using profiling cells.

In [ ]:
# Create a basic data profiling report
import pandas as pd
import numpy as np

def create_basic_profile(df):
    profile = {}
    
    # Basic DataFrame info
    profile['shape'] = df.shape
    profile['columns'] = df.columns.tolist()
    
    # Data types
    profile['dtypes'] = df.dtypes.to_dict()
    
    # Missing values
    profile['missing'] = df.isnull().sum().to_dict()
    
    # Basic statistics for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    profile['numeric_stats'] = {}
    
    for col in numeric_cols:
        profile['numeric_stats'][col] = {
            'count': df[col].count(),
            'mean': df[col].mean(),
            'std': df[col].std(),
            'min': df[col].min(),
            '25%': df[col].quantile(0.25),
            '50%': df[col].quantile(0.50),
            '75%': df[col].quantile(0.75),
            'max': df[col].max()
        }
    
    # Unique values for categorical columns
    profile['unique_counts'] = df.nunique().to_dict()
    
    return profile

# Generate the profile
df = pd.read_csv("resturent.csv")
profile = create_basic_profile(df)

# Print the results in a readable format
print("=== Data Profile Report ===\n")
print(f"Dataset Shape: {profile['shape'][0]} rows, {profile['shape'][1]} columns\n")

print("=== Column Types ===")
for col, dtype in profile['dtypes'].items():
    print(f"{col}: {dtype}")
print()

print("=== Missing Values ===")
missing_cols = {col: missing for col, missing in profile['missing'].items() if missing > 0}
if missing_cols:
    for col, missing in missing_cols.items():
        pct = (missing / profile['shape'][0]) * 100
        print(f"{col}: {missing} missing values ({pct:.2f}%)")
else:
    print("No missing values found in the dataset")
print()

print("=== Unique Values ===")
for col, count in profile['unique_counts'].items():
    pct = (count / profile['shape'][0]) * 100
    print(f"{col}: {count} unique values ({pct:.2f}%)")
print()

print("=== Numeric Statistics ===")
for col, stats in profile['numeric_stats'].items():
    print(f"\n{col}:")
    for stat, value in stats.items():
        print(f"  {stat}: {value:.2f}")